# Relative Humidity Calculations

In [ ]:
from CoolProp.HumidAirProp import HAPropsSI

def berechne_rel_feuchte():
    # Umgebungsdruck in Pascal (Standardatmosphäre)
    p_atm = 101325  
    
    # Werte aus der Tabelle: Betriebs-Nennbedingungen (Außenluft)
    # Format: (Trockenkugeltemperatur in °C, Feuchtkugeltemperatur in °C)
    bedingungen = [
        (12, 11),
        (7, 6),
        (2, 1),
        (-7, -8),
        (-10, -11) 
    ]
    
    print("Berechnung der relativen Luftfeuchtigkeit (Betriebs-Nennbedingungen)")
    print("-" * 70)
    print(f"{'Trockenkugel':<15} | {'Feuchtkugel':<15} | {'Relative Luftfeuchtigkeit'}")
    print("-" * 70)
    
    for t_dry, t_wet in bedingungen:
        # Umrechnung von Celsius in Kelvin für CoolProp
        t_dry_k = t_dry + 273.15
        t_wet_k = t_wet + 273.15
        
        try:
            # HAPropsSI-Parameter:
            # 'R' = gesuchter Wert: Relative Luftfeuchtigkeit (als Dezimalzahl)
            # 'T' = Trockenkugeltemperatur in Kelvin
            # 'B' = Feuchtkugeltemperatur in Kelvin (Wet-Bulb)
            # 'P' = Druck in Pascal
            rel_humidity_decimal = HAPropsSI('R', 'T', t_dry_k, 'B', t_wet_k, 'P', p_atm)
            
            # Umrechnung in Prozent
            rel_humidity_percent = rel_humidity_decimal * 100
            
            print(f"{t_dry:>10} °C   | {t_wet:>10} °C   | {rel_humidity_percent:>20.2f} %")
            
        except ValueError as e:
            print(f"Fehler bei {t_dry}°C / {t_wet}°C: {e}")

if __name__ == "__main__":
    berechne_rel_feuchte()

# Time Step Finding

In [ ]:
import os
import sys
import time
import pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from vclibpy.media import RefProp
from frost_evaporator import HeatPumpSimulation, FrostEvaporatorParameters

# =============================================================================
# 0. REFPROP CONFIGURATION
# =============================================================================
REFPROP_DIR = Path(r"C:\Program Files (x86)\REFPROP")
REFPROP_DLL = REFPROP_DIR / "REFPRP64.DLL"
os.environ["RPPREFIX"] = str(REFPROP_DIR)

# =============================================================================
# 1. SETUP & KONFIGURATION
# =============================================================================
CP_WATER = 4184             # Spezifische Wärmekapazität Wasser J/(kg K)
SIMULATION_DURATION = 12 * 60 * 60  # Maximales Abbruchkriterium (z.B. 12h)

# Define Configurations for the Heat Pump runs (similar to the evaporator script)
configs = [
    {
        'experiment_name': 'Point_B_Low',
        'evap_config': "config_OptiHorst.yaml",
        'fin_pitch': 0.00175,
        't_air': 2.0,                    # SCOP Point B, Low
        'rh_air': 84.0,                  # SCOP Point B, Low
        'q_cond_target': 10000 * 0.5385, # SCOP Point B, Low
        't_water_out': 34.0,             # SCOP Point B, Low
        't_water_in': 29.0               # SCOP Point B, Low
    }
    # You can easily add more dictionaries here for Point B, Point C, etc.
]

time_steps_to_test = [240, 120, 60, 30, 10, 5, 1]
all_combined_results = {}

# =============================================================================
# 2. SIMULATION LOOP
# =============================================================================
for conf in configs:
    exp_name = conf['experiment_name']
    evap_config = conf['evap_config']
    fin_pitch = conf['fin_pitch']
    q_target = conf['q_cond_target']
    
    # Calculate required water mass flow
    m_flow_water = q_target / (CP_WATER * (conf['t_water_out'] - conf['t_water_in']))

    # Load parameters to determine fin amount and initialize RefProp
    params = FrostEvaporatorParameters.from_yaml(evap_config)
    EVAP_WIDTH = params.fin_amount * params.fin_pitch
    fin_amount_calc = int(EVAP_WIDTH / fin_pitch)

    # Initialize RefProp based on refrigerant
    if params.refrigerant == "R410a":
        med_prop = RefProp(
            fluid_name="R32.FLD|R125.FLD",       
            z=[0.697615, 0.302385],
            dll_path=str(REFPROP_DLL), 
            ref_prop_path=str(REFPROP_DIR),
            copy_dll=False
        )
    elif params.refrigerant == "R134a":
        med_prop = RefProp(
            fluid_name="R134a",
            dll_path=str(REFPROP_DLL), 
            ref_prop_path=str(REFPROP_DIR),
            copy_dll=False
        )
    else:
        raise ValueError(f"Unsupported refrigerant: {params.refrigerant}")

    print(f"\n{'='*80}")
    print(f"STARTING TIME STEP SENSITIVITY ANALYSIS for: {exp_name}")
    print(f"Config: {evap_config} | Pitch: {fin_pitch}m | Target Q: {q_target}W")
    print(f"Air: {conf['t_air']}°C, {conf['rh_air']}% | Water: {conf['t_water_in']}°C -> {conf['t_water_out']}°C")
    print(f"{'='*80}")

    results = []

    for i, dt in enumerate(time_steps_to_test):
        print(f"Starte Lauf {i+1}/{len(time_steps_to_test)} mit dt = {dt} s...")
        start_time = time.time()
        
        # Simulationsobjekt initialisieren (Created fresh each loop to prevent state bleed)
        heat_pump_sim = HeatPumpSimulation(
            evap_config_path       = evap_config,
            external_refprop       = med_prop,
            time_step_size         = dt,
            simulation_duration    = SIMULATION_DURATION,
            t_air_in               = conf['t_air'], 
            rh_in                  = conf['rh_air'], 
            t_water_in             = conf['t_water_in'],
            q_cond_target          = q_target,
            m_flow_water           = m_flow_water
        )
        
        heat_pump_sim.update_and_prime_evaporator(
            fin_pitch  = fin_pitch,
            fin_amount = fin_amount_calc,
        )
        
        # Simulation ausführen
        res, end_reason, total_duration_min = heat_pump_sim.run_simulation()
        
        # Werte extrahieren
        final_frost_mass_kg = res["m_frost"][-1]
        final_dp_pa         = res["dp_air"][-1]
        final_q_tot_w       = res["Q_cond"][-1] 
        
        elapsed_time = time.time() - start_time
                
        results.append({
            'run': i + 1,
            'dt': dt,
            'mass': final_frost_mass_kg, 
            'dp': final_dp_pa, 
            'q': final_q_tot_w, 
            'duration_min': total_duration_min,
            'sim_time': elapsed_time
        })

    # =========================================================================
    # 3. AUSWERTUNG & TABELLEN-AUSGABE
    # =========================================================================
    ref_run = next(res for res in results if res['dt'] == min(time_steps_to_test))
    ref_mass, ref_dp, ref_q, ref_dur = ref_run['mass'], ref_run['dp'], ref_run['q'], ref_run['duration_min']

    table_width = 125 
    print("\n" + "=" * table_width)
    print(f"{'Lauf':<5} | {'dt':<4} | {'Reifmasse':<10} | {'Abw.%':<7} | {'Druckverl.':<10} | {'Abw.%':<7} | {'Wärmestrom':<10} | {'Abw.%':<7} | {'Dauer[min]':<10} | {'Abw.%':<7} | {'Sim.Zeit':<8}")
    print("-" * table_width)

    for res in results:
        res['dev_mass'] = ((res['mass'] - ref_mass) / ref_mass * 100) if ref_mass else 0.0
        res['dev_dp']   = ((res['dp'] - ref_dp) / ref_dp * 100) if ref_dp else 0.0
        res['dev_q']    = ((res['q'] - ref_q) / ref_q * 100) if ref_q else 0.0
        res['dev_dur']  = ((res['duration_min'] - ref_dur) / ref_dur * 100) if ref_dur else 0.0
        
        run, dt_str = res['run'], f"{res['dt']}s"
        mass_str, dp_str, q_str = f"{res['mass']:.4f}", f"{res['dp']:.2f}", f"{res['q']:.2f}"
        dur_str = f"{res['duration_min']:.2f}"
        time_str = f"{res['sim_time']:.2f}s"
        
        if res['dt'] == min(time_steps_to_test):
            dm_str, ddp_str, dq_str, ddur_str = "-", "-", "-", "-"
        else:
            dm_str   = f"{res['dev_mass']:+5.2f}%"
            ddp_str  = f"{res['dev_dp']:+5.2f}%"
            dq_str   = f"{res['dev_q']:+5.2f}%"
            ddur_str = f"{res['dev_dur']:+5.2f}%"
            
        print(f"{run:<5} | {dt_str:<4} | {mass_str:<10} | {dm_str:<7} | {dp_str:<10} | {ddp_str:<7} | {q_str:<10} | {dq_str:<7} | {dur_str:<10} | {ddur_str:<7} | {time_str:<8}")

    print("=" * table_width + "\n")
    
    # Store results for this experiment type to match the dictionary structure of the first script
    all_combined_results[exp_name] = results

# =============================================================================
# 4. DATEN SPEICHERN
# =============================================================================
save_dir = Path("simulation_data/final")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "heatpump_timestep_study_test.pkl"

print(f"Speichere kombinierte Ergebnisse in {save_path}...")
with open(save_path, "wb") as f:
    pickle.dump(all_combined_results, f)

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

# =============================================================================
# 1. GLOBALE PLOT-EINSTELLUNGEN (Vorgaben für Masterarbeit)
# =============================================================================
cm_to_inch = 1 / 2.54
width_cm = 15.5
# 3:2 Verhältnis als harmonischer Standard
height_cm = width_cm * (2/4)

plt.rcParams.update({
    # Maße und Auflösung
    "figure.figsize": (width_cm * cm_to_inch, height_cm * cm_to_inch),
    "figure.dpi": 300,
    
    # Schriftarten und LaTeX-Integration
    "text.usetex": True,
    # icomma sorgt dafür, dass das Komma in Matheumgebungen keinen falschen Abstand erzeugt
    "text.latex.preamble": r"\usepackage{icomma} \usepackage{amsmath}",
    "font.family": "serif",
    
    # Exakte Schriftgrößen laut Vorgaben
    "font.size": 11,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    
    # Linien
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    
    # Legende und Export
    "legend.frameon": False,
    "savefig.bbox": "tight",
    "savefig.format": "pdf"
})

# ERC / RWTH Farbpalette
colors = {
    "blue": "#00549F",      # Druckverlust
    "red": "#DD402D",       # Wärmestrom
    "green": "#70AD47",     # Reifmasse
    "dark_grey": "#4E4F50", # Simulationszeit / Referenzlinie
    "mid_grey": "#9D9EA0",  # Toleranzbänder
    "light_grey": "#D9D9D9" # Grid
}

# =============================================================================
# 2. HILFSFUNKTIONEN
# =============================================================================
def german_formatter(x, pos):
    r"""
    Formatiert die Achsenticks ins deutsche Format (Komma statt Punkt).
    Nutzt den TeX Math-Mode, welcher durch \usepackage{icomma} sauber gesetzt wird.
    """
    return f"${x:g}$".replace('.', ',')

# =============================================================================
# 3. DATEN LADEN UND PLOTTEN
# =============================================================================
# Update the path to point to your newly generated heat pump data
save_path = Path("simulation_data/final/06_heatpump_timestep_study_final.pkl")

print(f"Loading data from {save_path} for plotting...\n")
with open(save_path, "rb") as f:
    loaded_results = pickle.load(f)

m_style, m_size = 'o', 4

for exp_type, data in loaded_results.items():
    # Filtert alle Einträge mit dt = 480 raus (falls vorhanden)
    data = [d for d in data if d['dt'] != 480]

    print(f"Generating zeitschrittstudie plot for Heat Pump ({exp_type})...")
    
    dt_labels = [str(d['dt']) for d in data]
    reifmasse = np.array([d['mass'] for d in data])
    druckverlust = np.array([d['dp'] for d in data])
    waermestrom = np.array([d['q'] for d in data])
    sim_zeit = [d['sim_time'] for d in data]
    
    # Normierung in Prozent (Referenz ist das letzte Element in der Liste, hier typischerweise dt=1)
    reif_norm = reifmasse / reifmasse[-1] * 100
    druck_norm = druckverlust / druckverlust[-1] * 100
    waerme_norm = waermestrom / waermestrom[-1] * 100
    
    # Umwandeln der Simulationszeit in Minuten
    sim_zeit = [t / 60.0 for t in sim_zeit]
    
    x = np.arange(len(dt_labels))
    
    # Plot aufbauen
    fig, ax1 = plt.subplots()
    
    # Grid nur auf der y-Achse (Z-Order = 0 für Hintergrund)
    ax1.grid(True, axis='y', linestyle='--', color=colors["light_grey"], alpha=0.7, zorder=0)
    ax1.set_axisbelow(True) # Zwingt das Grid in den Hintergrund
    
    # Rahmenlinien (Spines) entfernen (Oben komplett, rechts für ax1)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    # Primäre Achse: Daten zeichnen (zorder=3 sorgt dafür, dass sie VOR dem Grid liegen)
    line1 = ax1.plot(x, reif_norm, marker=m_style, markersize=m_size, color=colors["green"], zorder=3, label='Reifmasse')
    line2 = ax1.plot(x, druck_norm, marker=m_style, markersize=m_size, color=colors["blue"], zorder=3, label='Druckverlust')
    line3 = ax1.plot(x, waerme_norm, marker=m_style, markersize=m_size, color=colors["red"], zorder=3, label='Wärmestrom')
    
    # Achsenbeschriftungen (Mit Escaping für Prozentzeichen in LaTeX)
    ax1.set_xlabel(r'Zeitschrittgröße $\Delta t$ [s]')
    ax1.set_ylabel('Normierte Endwerte\n' r'(Ref. $\Delta t = 1\,$s) [\%]')
    
    # Tick-Beschriftungen formatieren
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"${l}$" for l in dt_labels])
    ax1.yaxis.set_major_formatter(ticker.FuncFormatter(german_formatter))
    
    # Horizontale Linien für Referenz und Toleranz (zorder=1 -> Hinter den Daten, vor dem Grid)
    line_100 = ax1.axhline(100, color=colors["dark_grey"], linestyle='-', alpha=0.6, linewidth=1.2, zorder=1, label='Referenz (1 s)')
    line_tol = ax1.axhline(101, color=colors["mid_grey"], linestyle='--', alpha=0.8, linewidth=1.2, zorder=1, label=r'$\pm 1\,\%$ Grenze')
    ax1.axhline(99, color=colors["mid_grey"], linestyle='--', alpha=0.8, linewidth=1.2, zorder=1)
    ax1.axhspan(99, 101, color=colors["mid_grey"], alpha=0.1, zorder=0)
    
    # Sekundäre Y-Achse für Simulationszeit
    ax2 = ax1.twinx()
    # Auch hier oberen Rahmen entfernen
    ax2.spines['top'].set_visible(False)
    
    line4 = ax2.plot(x, sim_zeit, marker=m_style, markersize=m_size, color=colors["dark_grey"], linestyle=':', zorder=3, label='Simulationszeit')
    
    # Achsenbeschriftung für die rechte Achse
    ax2.set_ylabel(r'Simulationszeit [min]')
    
    # Tick-Parameter & Deutsches Nummernformat auf sekundärer Achse
    ax2.tick_params(axis='y')
    ax2.yaxis.set_major_formatter(ticker.FuncFormatter(german_formatter))
    
    # 1. Linke Y-Achse (ax1)
    ax1.set_ylim(94.5, 105.5) # Etwas Puffer, damit die Ticks nicht am Rand kleben
    ax1.set_yticks([95, 100, 105])
    
    # 2. Rechte Y-Achse (ax2)
    ax2.set_ylim(0, 120) 
    ax2.set_yticks([0, 30, 60, 90, 120])
    
    # Legende (Ausgelagert & Zentriert unterhalb des Plots)
    ordered_lines = [line2[0], line_100, line3[0], line_tol, line1[0], line4[0]]
    ordered_labels = [l.get_label() for l in ordered_lines]
    
    # bbox_to_anchor verschiebt die Legende nach UNTEN außerhalb der Axes-Fläche
    ax1.legend(ordered_lines, ordered_labels, loc='upper center', 
               bbox_to_anchor=(0.5, -0.3), ncol=3, frameon=False)
    
    # Rendern & Speichern
    plt.tight_layout()
    
    # Zielordner als Raw-String (wichtig wegen der Backslashes!)
    output_dir = r"J:\Masterarbeit_Nils_Baumeister\02_Schriftliche_Ausarbeitung\Graphiken"
    os.makedirs(output_dir, exist_ok=True) # Stelle sicher, dass der Ordner existiert
    
    # Dateiname generieren (wp = Wärmepumpe, experiment name included so files don't overwrite if you have multiple points)
    pdf_filename = f'6_zeitschritt_wp.pdf'
    
    # Ordner und Dateiname sicher zusammenfügen
    full_save_path = os.path.join(output_dir, pdf_filename)
    
    # Plot speichern
    plt.savefig(full_save_path, bbox_inches='tight')
    print(f"Plot erfolgreich gespeichert unter: {full_save_path}")

    plt.show()
    plt.close()

# SCOP Data Generation

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pickle
import yaml

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from vclibpy.media import RefProp
from frost_evaporator import HeatPumpSimulation, FrostEvaporatorParameters

# =============================================================================
# REFPROP CONFIGURATION
# =============================================================================
REFPROP_DIR = Path(r"C:\Program Files (x86)\REFPROP")
REFPROP_DLL = REFPROP_DIR / "REFPRP64.DLL"
os.environ["RPPREFIX"] = str(REFPROP_DIR)


def calculate_cycle_COP(run_data, dt_seconds): 
    """
    Calculates the integrated net COP, average heating power, and specific average 
    electrical power consumption (compressor, fan, and defrost).
    
    Defrost energy is amortized over the total cycle runtime.
    
    Args:
        run_data (dict): Dictionary containing operational data arrays.
        dt_seconds (float): Time step in seconds.
        
    Returns:
        dict: A dictionary containing the COP and averaged power/heat values.
    """
    
    # --- Constants ---
    ETA_DEFROST_HEATER = 0.45    # Electrical Heater doi: 10.1016/j.applthermaleng.2023.121072
    CP_ICE = 2100.0              # J/kg*K
    H_FUSION = 334000.0          # J/kg (Latent heat)

    # --- Extract Operational Arrays ---
    q_cond_arr = np.array(run_data["Q_cond"])
    p_comp = np.array(run_data["compressor_power"])
    p_fan  = np.array(run_data["fan_power"])
    
    # --- Calculate Total Time ---
    total_time = len(q_cond_arr) * dt_seconds

    # --- Calculate Operational Energy (Joules) ---
    E_heating_useful = np.sum(q_cond_arr) * dt_seconds
    E_comp = np.sum(p_comp) * dt_seconds
    E_fan = np.sum(p_fan) * dt_seconds

    # --- Calculate Defrost Energy ---
    E_defrost_thermal = 0.0
    layer_keys = [k for k in run_data.keys() if k.startswith("m_frost_L")]
    
    for key_mass in layer_keys:
        layer_suffix = key_mass.split("_")[-1]
        key_temp = f"T_frost_avg_{layer_suffix}"
        
        # Taking the final mass and temperature before defrost
        m_final = run_data[key_mass][-1]
        t_final = run_data[key_temp][-1] - 273.15   # Convert to Celsius
        
        # Energy to heat ice to 0°C and melt it
        q_sensible = m_final * CP_ICE * (0.0 - t_final)
        q_latent   = m_final * H_FUSION
        E_defrost_thermal += (q_sensible + q_latent)

    # Convert thermal energy to electrical energy required by the heater
    E_defrost_electrical = E_defrost_thermal / ETA_DEFROST_HEATER

    # --- Calculate Average Powers (amortized over total runtime) ---
    actual_q_dot     = E_heating_useful / total_time
    actual_p_comp    = E_comp / total_time
    actual_p_fan     = E_fan / total_time
    actual_p_defrost = E_defrost_electrical / total_time
    
    # Total average electrical power
    actual_p_elec = actual_p_comp + actual_p_fan + actual_p_defrost

    # --- Calculate Final Efficiency (COP) ---
    # Safe division to avoid ZeroDivisionError just in case
    cop = actual_q_dot / actual_p_elec if actual_p_elec > 0 else 0.0

    # --- Return Results ---
    return {
        "cop": cop,
        "actual_q_dot": actual_q_dot,
        "actual_p_elec_total": actual_p_elec,
        "actual_p_comp": actual_p_comp,
        "actual_p_fan": actual_p_fan,
        "actual_p_defrost": actual_p_defrost
    }



# =============================================================================
# Individual Settings
# =============================================================================

# EVAP_CONFIG = "config_OptiHorst.yaml"
# P_DESIGNH = 10000  # Defined by Preliminary Study (E, MEDIUM-BECAUSE CRITICAL POINT, fin_pitch = 3mm, compressor_speed = 0.9)
# # pitches = [0.00075, 0.001, 0.00125, 0.0015, 0.00175, 0.002, 0.00225, 0.0025, 0.00275, 0.003]
# pitches = [0.001, 0.00125, 0.0015, 0.00175, 0.002, 0.00225, 0.0025, 0.00275, 0.003]


EVAP_CONFIG = "config_OptiAbt.yaml"
P_DESIGNH = 10400  # Defined by Preliminary Study (E, High, fin_pitch = 8mm, compressor_speed = 0.9)
pitches = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008]

# P_DESIGNH = 13000.22609  # 11500 / 0.8846 = 13000.22609 W -> So dass im Versuch genau die 11,5kW abgerufen werden   (A, High, fin_pitch = 8mm, compressor_speed = 0.9)
# P_DESIGNH = 27855.1532   # 15000 / 0.5385 = 27855.1532 W  -> So dass im Versuch genau die 15kW   abgerufen werden   (B, High, fin_pitch = 8mm, compressor_speed = 0.9)
# pitches = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008]


# Simulation Time
TIME_STEP_SIZE = 10  # seconds
SIMULATION_DURATION = 12 * 60 * 60 # 24h Maximum



# =============================================================================
# General Settings
# =============================================================================

params = FrostEvaporatorParameters.from_yaml(EVAP_CONFIG)
EVAP_WIDTH = params.fin_amount * params.fin_pitch

if params.refrigerant == "R410a":
    med_prop = RefProp(
        fluid_name="R32.FLD|R125.FLD",       
        z=[0.697615, 0.302385],
        dll_path=str(REFPROP_DLL), 
        ref_prop_path=str(REFPROP_DIR),
        copy_dll=False
    )
elif params.refrigerant == "R134a":
    med_prop = RefProp(
        fluid_name="R134a",
        dll_path=str(REFPROP_DLL), 
        ref_prop_path=str(REFPROP_DIR),
        copy_dll=False
    )
else:
    raise ValueError(f"Unsupported refrigerant: {params.refrigerant}")


# Test points (Mittleres Klima). RH calculated from norm Wet-Bulb temps.
test_points = {
    'E': {'t_air': -10.0, 'rh': 69.0, 'plr': 1.0000},
    'A': {'t_air':  -7.0, 'rh': 75.0, 'plr': 0.8846},
    'B': {'t_air':   2.0, 'rh': 84.0, 'plr': 0.5385},
    'C': {'t_air':   7.0, 'rh': 87.0, 'plr': 0.3462},
    'D': {'t_air':  12.0, 'rh': 89.0, 'plr': 0.1538},
}

# Water outlet temperatures for each test point and application.
water_out_temps = {
    'Low':    {'A': 34, 'B': 30, 'C': 27, 'D': 24, 'E': 35},
    # 'Inter':  {'A': 43, 'B': 37, 'C': 33, 'D': 28, 'E': 45},
    # 'Medium': {'A': 52, 'B': 42, 'C': 36, 'D': 30, 'E': 55},
    # 'High':   {'A': 61, 'B': 49, 'C': 41, 'D': 32, 'E': 65} 
}

CP_WATER = 4184  # Specific heat capacity of water in J/(kg K)

# Norm-specific temperature spread (Delta T) in K
dt_spread = {
    'Low': 5.0,
    'Inter': 5.0,
    'Medium': 8.0,
    'High': 10.0
}

# =============================================================================
# BUILD RUN COMBINATIONS
# =============================================================================
combinations = []
for pitch in pitches:
    for app_name, pt_temps in water_out_temps.items():
        for pt_name, pt_data in test_points.items():
            
            # Calculate target condenser heat based on Part Load Ratio (PLR)
            q_cond_target = P_DESIGNH * pt_data['plr']
            
            # 1. Fetch the correct water outlet temp for this app and weather point
            t_out = pt_temps[pt_name]
            
            # 2. Calculate the inlet temp based on the norm's delta T for the specific application
            t_in = t_out - dt_spread[app_name]
            
            combinations.append({
                'pitch': pitch,
                'app_name': app_name,
                'pt_name': pt_name,
                't_air': pt_data['t_air'],
                'rh': pt_data['rh'],
                'q_cond': q_cond_target,
                't_water_out': t_out,
                't_water_in': t_in
            })

print(f"Starting SCOP Matrix Simulation: {len(combinations)} combinations.\n")




# =============================================================================
# SIMULATION LOOP
# =============================================================================
all_runs_data = [] 

for i, c in enumerate(combinations):
    run_id = i + 1
    print(f"\n\n--- Run {run_id}/{len(combinations)} | Pitch={c['pitch']:.5f} | App={c['app_name']} | Point={c['pt_name']} ---")

    # Calculate required water mass flow rate (kg/s) based on original target
    original_q_cond = c['q_cond']
    m_flow_water = original_q_cond / (CP_WATER * (c['t_water_out'] - c['t_water_in']))

    try:
        # Determine fin amount
        amount = int(round(EVAP_WIDTH / c['pitch']))

        # =========================================================================
        # PRELIMINARY BOUNDS CHECK
        # =========================================================================

        print("--- Bounds Check --- ")
        
        # Create safe mass flows for the bounds checks to prevent supercritical crashes.
        # We assume max capacity is roughly 120% of P_DESIGNH and min is roughly 30%.
        current_dt = dt_spread[c['app_name']]
        
        q_max_safe_guess = P_DESIGNH * 1.5
        m_flow_max_safe = q_max_safe_guess / (CP_WATER * current_dt)
        
        # 2. Min Safe Guess
        # The Simplest Way: Use the nominal design mass flow for the bounds check.
        # This guarantees the water won't overheat, preventing high-pressure crashes.
        m_flow_min_safe = P_DESIGNH / (CP_WATER * current_dt)

        # 1. Check Min Bounds (Speed = 0.3)
        sim_min = HeatPumpSimulation(
            evap_config_path       = EVAP_CONFIG,
            external_refprop       = med_prop,
            time_step_size         = TIME_STEP_SIZE,
            simulation_duration    = TIME_STEP_SIZE * 10, # Run 10 time-steps to converge!
            t_air_in               = c['t_air'], 
            rh_in                  = c['rh'], 
            t_water_in             = c['t_water_in'],
            q_cond_target          = None,
            m_flow_water           = m_flow_min_safe,
        )
        sim_min.fixed_compressor_speed = sim_min.comp_min_rel_speed + 0.03 # Add small Buffer to stay above min speed
        sim_min.update_and_prime_evaporator(fin_pitch=c['pitch'], fin_amount=amount)
        res_min, _, _ = sim_min.run_simulation()
        q_cond_array = res_min['Q_cond']
        q_min = sum(q_cond_array[-3:]) / 3.0

        # # Check Max Bounds (90% speed for frost capability)
        # sim_max = HeatPumpSimulation(
        #     evap_config_path       = EVAP_CONFIG,
        #     external_refprop       = med_prop,
        #     time_step_size         = TIME_STEP_SIZE,
        #     simulation_duration    = TIME_STEP_SIZE, # Run exactly 1 time-step!
        #     t_air_in               = c['t_air'], 
        #     rh_in                  = c['rh'], 
        #     t_water_in             = c['t_water_in'],
        #     q_cond_target          = None,
        #     m_flow_water           = m_flow_max_safe, 
        #     fixed_compressor_speed = 0.9
        # )

        
        # sim_max.update_and_prime_evaporator(fin_pitch=c['pitch'], fin_amount=amount)
        # res_max, _, _ = sim_max.run_simulation()
        # q_max = res_max['Q_cond'][0]
        # print(q_max)

        # 3. Apply Caps
        actual_q_cond_target = original_q_cond
        capped_marker = "No Capping Needed"
        
        if original_q_cond < q_min:
            actual_q_cond_target = q_min
            capped_marker = "Capped at Minimum"

            
        print(f"  -> Bounds Check: Q_min={q_min:.1f}W")
        print(f"  -> Target={original_q_cond:.1f}W  ==>  {capped_marker} ({actual_q_cond_target:.1f}W)")



        # =========================================================================
        # ACTUAL SIMULATION RUN
        # =========================================================================

        print("--- Actual Simulation --- ")

        actual_m_flow_water = actual_q_cond_target / (CP_WATER * current_dt)

        heat_pump_sim = HeatPumpSimulation(
            evap_config_path       = EVAP_CONFIG,
            external_refprop       = med_prop,
            time_step_size         = TIME_STEP_SIZE,
            simulation_duration    = SIMULATION_DURATION,
            t_air_in               = c['t_air'], 
            rh_in                  = c['rh'], 
            t_water_in             = c['t_water_in'],
            q_cond_target          = actual_q_cond_target, # Feed the capped target!
            m_flow_water           = actual_m_flow_water
            # NO fixed_compressor_speed here!
        )
        
        # Update geometry and prime the model
        heat_pump_sim.update_and_prime_evaporator(
            fin_pitch  = c['pitch'],
            fin_amount = amount,
        )
        
        # Run Simulation
        res, end_reason, total_duration_min = heat_pump_sim.run_simulation()

        cop_results = calculate_cycle_COP(res, dt_seconds=TIME_STEP_SIZE)

        # Tag the result with structured inputs for easier evaluation later
        res["metadata"] = {
            "run_id": run_id,
            "norm_context": {
                "application": c['app_name'],
                "test_point": c['pt_name']
            },
            "inputs": {
                "q_cond_original":    original_q_cond,
                "q_cond_applied":     actual_q_cond_target,
                "capped_marker":      capped_marker,
                "q_min_bound":        q_min,
                "t_water_in":         c['t_water_in'],
                "t_water_out":        c['t_water_out'],
                "m_flow_water":       actual_m_flow_water,
                "t_air":              c['t_air'],
                "rh":                 c['rh'],
                "pitch":              c['pitch'],
                "amount":             amount,
                "cycle-COP":          cop_results["cop"],
                "actual_q_dot":       cop_results["actual_q_dot"],
                "actual_p_elec":      cop_results["actual_p_elec_total"],
                "actual_p_comp":      cop_results["actual_p_comp"],
                "actual_p_fan":       cop_results["actual_p_fan"],
                "actual_p_defrost":   cop_results["actual_p_defrost"],
                "end_reason":         end_reason,
                "total_duration_min": total_duration_min
            }
        }

        # Store in Master List
        all_runs_data.append(res)

    except Exception as e:
        print(f"!!! Error on Run {run_id}: {e}")
        # Append error info so we don't lose track of which run failed
        all_runs_data.append({
            "run_id": run_id, 
            "error": str(e),
            "metadata": {"inputs": c, "norm_context": {"application": c['app_name'], "test_point": c['pt_name']}}
        })

# =============================================================================
# SAVE RESULTS
# =============================================================================
output_filename = "simulation_data/grand_scop_simulation_results_abt_betta_stuff.pkl"
print(f"\nSaving all data to '{output_filename}'...")

# Pack the simulation data AND the boundary conditions into one payload
save_payload = {
    "runs_data": all_runs_data,
    "test_points": test_points,
    "water_out_temps": water_out_temps,
    "P_DESIGNH": P_DESIGNH
}

with open(output_filename, "wb") as f:
    pickle.dump(save_payload, f)
    
print("Done.")

Starting SCOP Matrix Simulation: 40 combinations.



--- Run 1/40 | Pitch=0.00100 | App=Low | Point=E ---
--- Bounds Check --- 


Sim Time 2min: 100%|██████████| 10/10 [00:32<00:00,  3.26s/step, P_evap=1.40bar,   |  dH_Err=18.3 J/kg,   |  Full Iterations=3,   |  Speed=0.39,   |  COP Ratio=-]    


  -> Bounds Check: Q_min=5460.5W
  -> Target=10400.0W  ==>  No Capping Needed (10400.0W)
--- Actual Simulation --- 


Sim Time 50min:   7%|▋         | 303/4320 [04:04<53:55,  1.24step/s, P_evap=1.07bar,   |  dH_Err=49.3 J/kg,   |  Full Iterations=3,   |  Speed=0.94,   |  COP Ratio=93.6%]    



[STOP] COP dropped to 93.5% of baseline value (step 10). Defrost triggered.


--- Run 2/40 | Pitch=0.00100 | App=Low | Point=A ---
--- Bounds Check --- 


...



--- Run 40/40 | Pitch=0.00800 | App=Low | Point=D ---
--- Bounds Check --- 


Sim Time 2min: 100%|██████████| 10/10 [00:16<00:00,  1.68s/step, P_evap=2.84bar,   |  dH_Err=-1.0 J/kg,   |  Full Iterations=2,   |  Speed=0.39,   |  COP Ratio=-]    


  -> Bounds Check: Q_min=10378.9W
  -> Target=1599.5W  ==>  Capped at Minimum (10378.9W)
--- Actual Simulation --- 


Sim Time 5min:   1%|          | 30/4320 [00:19<45:28,  1.57step/s, P_evap=2.84bar,   |  dH_Err=-0.1 J/kg,   |  Full Iterations=2,   |  Speed=0.39,   |  COP Ratio=100.0%] 



[STOP] Early Stoppage: No Frost Growth detected.

Saving all data to 'simulation_data/grand_scop_simulation_results_abt_betta_stuff.pkl'...
Done.


# Individual Plots

In [ ]:
import os
import sys
import pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import ScalarFormatter

# =============================================================================
# REFPROP CONFIGURATION (Needed for the Saturation Dome)
# =============================================================================
from vclibpy.media import RefProp

REFPROP_DIR = Path(r"C:\Program Files (x86)\REFPROP")
REFPROP_DLL = REFPROP_DIR / "REFPRP64.DLL"
os.environ["RPPREFIX"] = str(REFPROP_DIR)


# =============================================================================
# DASHBOARD PLOTTING LOGIC
# =============================================================================
def plot_dashboard(results, med_prop):
    # Set Style
    try:
        plt.style.use('seaborn-v0_8')
    except:
        plt.style.use('ggplot')

    # Create Figure (Wider to accommodate 4 columns)
    fig = plt.figure(figsize=(28, 16))
    
    # Grab metadata if available to put in the title
    title_suffix = ""
    if "metadata" in results and "inputs" in results["metadata"]:
        inputs = results["metadata"]["inputs"]
        title_suffix = f" | Pitch: {inputs.get('pitch')}m, Q_cond: {inputs.get('q_cond')}W, T_air: {inputs.get('t_air')}°C"
        
    fig.suptitle(f'Frosting Heat Pump Cycle Simulation - Full Dashboard{title_suffix}', fontsize=20, fontweight='bold')

    # Create Grid Spec (3 Rows x 4 Columns)
    gs = gridspec.GridSpec(3, 4, wspace=0.25, hspace=0.3)
    
    time_min = np.array(results["time"])

    # =============================================================================
    # ROW 1: [0,0] logph | [0,1] COP | [0,2] Power | [0,3] Solver Iterations
    # =============================================================================

    # 1. log(p)-h Diagram (0,0)
    ax1 = plt.subplot(gs[0, 0])
    
    # -- Saturation Dome Logic --
    try:
        if hasattr(med_prop, 'get_critical_point'):
             _, p_crit, _ = med_prop.get_critical_point()
        else:
             p_crit = 4900000.0 
        
        p_dome_range = np.logspace(np.log10(1e5), np.log10(p_crit * 0.99), 100)
        h_liq_dome, h_vap_dome, p_dome_success = [], [], []

        for p in p_dome_range:
            try:
                hl = med_prop.calc_state("PQ", p, 0.0).h / 1000.0
                hv = med_prop.calc_state("PQ", p, 1.0).h / 1000.0
                h_liq_dome.append(hl)
                h_vap_dome.append(hv)
                p_dome_success.append(p)
            except: pass

        h_dome_full = np.concatenate([h_liq_dome, h_vap_dome[::-1]])
        p_dome_full = np.concatenate([p_dome_success, p_dome_success[::-1]])
        ax1.plot(h_dome_full, p_dome_full / 1e5, 'k-', linewidth=1, alpha=0.5, label='Sat. Curve')
    except: pass

    # -- Plot Cycles --
    colors = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd']
    cycles_to_plot = results.get("cycles", []) 
    
    for idx, cycle in enumerate(cycles_to_plot):
        h_cycle_kJ = np.array(cycle["h"]) / 1000.0
        p_cycle_bar = np.array(cycle["p"]) / 1e5
        c_color = colors[idx % len(colors)]
        ax1.plot(h_cycle_kJ, p_cycle_bar, color=c_color, linewidth=2, 
                 label=cycle.get("name", f"Cycle {idx}"), marker='.', markersize=8)

    ax1.set_title('log(p)-h Diagram', fontweight='bold')
    ax1.set_ylabel('Pressure [bar]')
    ax1.set_xlabel('Enthalpy [kJ/kg]')
    ax1.set_yscale('log')
    ax1.yaxis.set_major_formatter(ScalarFormatter())
    ax1.grid(True, which="both", ls="--", alpha=0.4)
    ax1.legend(loc='best', fontsize='x-small')

    # 2. COP (0,1)
    ax2 = plt.subplot(gs[0, 1])
    ax2.plot(time_min, results["COP_wo_defrost"], color='#ff7f0e', linewidth=2)
    ax2.set_title('Coefficient of Performance (COP)', fontweight='bold')
    ax2.set_ylabel('COP [-]')
    ax2.grid(True, linestyle='--', alpha=0.7)
    ax2.tick_params(labelbottom=False)

    # 3. Power (0,2)
    ax3 = plt.subplot(gs[0, 2])
    ax3.plot(time_min, np.array(results["Q_cond"])/1000, label='$Q_{cond}$', color='#d62728')
    ax3.plot(time_min, np.array(results["Q_evap"])/1000, label='$Q_{evap}$', color='#1f77b4')
    ax3.plot(time_min, np.array(results["compressor_power"])/1000, label='$P_{el}$', color='#2ca02c')
    ax3.set_title('Thermal & Electrical Power', fontweight='bold')
    ax3.set_ylabel('Power [kW]')
    ax3.grid(True, linestyle='--', alpha=0.7)
    ax3.legend(loc='best', fontsize='small')
    ax3.tick_params(labelbottom=False)

    # 4. Solver Iterations (0,3)
    gs_inner = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[0, 3], hspace=0.1)
    ax4_top = plt.subplot(gs_inner[0, 0])
    ax4_bot = plt.subplot(gs_inner[1, 0], sharex=ax4_top)

    ax4_top.plot(time_min, results["iterations"]["Level 3"], label='Lvl 3 (Evap)', color='tab:orange', linewidth=2)
    ax4_top.set_title('Solver Iterations', fontweight='bold')
    ax4_top.set_ylabel('Lvl 3 [-]')
    ax4_top.grid(True, linestyle='--', alpha=0.7)
    ax4_top.tick_params(labelbottom=False)
    
    ax4_bot.plot(time_min, results["iterations"]["Level 2"], label='Lvl 2 (Cond)', color='tab:blue', alpha=0.7)
    ax4_bot.set_ylabel('Lvl 2 [-]')
    ax4_bot.grid(True, linestyle='--', alpha=0.7)
    ax4_bot.tick_params(labelbottom=False)

    # =============================================================================
    # ROW 2: [1,0] speed | [1,1] ref massflow | [1,2] pressures | [1,3] h1
    # =============================================================================

    # 5. Speed (1,0)
    ax5 = plt.subplot(gs[1, 0])
    ax5.plot(time_min, results["speed"], color='#8c564b', linewidth=2)
    ax5.set_title('Compressor Speed', fontweight='bold')
    ax5.set_ylabel('Speed [Rel. / Hz]') 
    ax5.grid(True, linestyle='--', alpha=0.7)
    ax5.tick_params(labelbottom=False)

    # 6. Ref Mass Flow (1,1)
    ax6 = plt.subplot(gs[1, 1])
    ax6.plot(time_min, results["m_flow"], color='#9467bd', linewidth=2)
    ax6.set_title('Ref. Mass Flow', fontweight='bold')
    ax6.set_ylabel('Mass Flow [kg/s]')
    ax6.grid(True, linestyle='--', alpha=0.7)
    ax6.tick_params(labelbottom=False)

    # 7. System Pressures (1,2)
    ax7 = plt.subplot(gs[1, 2])
    ax7.plot(time_min, np.array(results["p_cond"])/1e5, label='$p_{cond}$', color='#d62728')
    ax7.plot(time_min, np.array(results["p_evap"])/1e5, label='$p_{evap}$', color='#1f77b4')
    ax7.set_title('System Pressures', fontweight='bold')
    ax7.set_ylabel('Pressure [bar]')
    ax7.grid(True, linestyle='--', alpha=0.7)
    ax7.legend(loc='center right', fontsize='small')
    ax7.tick_params(labelbottom=False)

    # 8. Evap Outlet Enthalpy (1,3)
    ax8 = plt.subplot(gs[1, 3])
    ax8.plot(time_min, np.array(results["h1"])/1000.0, color='#17becf', linewidth=2)
    ax8.set_title('Evap. Outlet Enthalpy ($h_1$)', fontweight='bold')
    ax8.set_ylabel('Enthalpy [kJ/kg]')
    ax8.grid(True, linestyle='--', alpha=0.7)
    ax8.tick_params(labelbottom=False)

    # =============================================================================
    # ROW 3: [2,0] volumeflow | [2,1] dp_air | [2,2] space_between | [2,3] m_frost
    # =============================================================================

    # 9. Air Volume Flow (2,0)
    ax9 = plt.subplot(gs[2, 0])
    ax9.plot(time_min, results["v_dot_air"], color='#2ca02c', linewidth=2)
    ax9.set_title('Air Volume Flow', fontweight='bold')
    ax9.set_ylabel('Flow [$m^3/h$]')
    ax9.set_xlabel('Time [min]')
    ax9.grid(True, linestyle='--', alpha=0.7)

    # 10. Pressure Drop (2,1)
    ax10 = plt.subplot(gs[2, 1])
    ax10.plot(time_min, results["dp_air"], color='#7f7f7f', linewidth=2)
    ax10.set_title('Air Side Pressure Drop', fontweight='bold')
    ax10.set_ylabel('$\Delta P_{air}$ [Pa]')
    ax10.set_xlabel('Time [min]')
    ax10.grid(True, linestyle='--', alpha=0.7)

    # 11. Space Between Frost (2,2)
    ax11 = plt.subplot(gs[2, 2])
    ax11.plot(time_min, np.array(results["space_between_frost"]) * 1000, color='#e377c2', linewidth=2)
    ax11.set_title('Free Space b/w Fins', fontweight='bold')
    ax11.set_ylabel('Distance [mm]') 
    ax11.set_xlabel('Time [min]')
    ax11.set_ylim(0, None)
    ax11.grid(True, linestyle='--', alpha=0.7)

    # 12. Frost Mass (2,3)
    ax12 = plt.subplot(gs[2, 3])
    ax12.plot(time_min, results["m_frost"], color='#bcbd22', linewidth=2)
    ax12.set_title('Accumulated Frost Mass', fontweight='bold')
    ax12.set_ylabel('Mass [kg]') 
    ax12.set_xlabel('Time [min]')
    ax12.grid(True, linestyle='--', alpha=0.7)

    # Final Layout Adjustments
    plt.tight_layout()
    plt.subplots_adjust(top=0.92) # Leave space for main title
    plt.show()

# =============================================================================
# DATA LOADING & EXECUTION
# =============================================================================
# med_prop = RefProp(
#     fluid_name="R32.FLD|R125.FLD",       
#     z=[0.697615, 0.302385],
#     dll_path=str(REFPROP_DLL), 
#     ref_prop_path=str(REFPROP_DIR),
#     copy_dll=False
# )
med_prop = RefProp(
    fluid_name="R134a",
    dll_path=str(REFPROP_DLL), 
    ref_prop_path=str(REFPROP_DIR),
    copy_dll=False
)

file_path = "simulation_data\grand_scop_simulation_results_abt_betta_test_12.pkl"
# file_path = "simulation_data\grand_scop_simulation_results_abt_test.pkl"

with open(file_path, "rb") as f:
    all_runs_data = pickle.load(f)


for i in range(8):
    target_result = all_runs_data["runs_data"][i]
    plot_dashboard(target_result, med_prop)


# target_result = all_runs_data["runs_data"][163]
# plot_dashboard(target_result, med_prop)

In [ ]:
import os
import sys
import pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# =============================================================================
# 1. GLOBALE FORMATIERUNGS-EINSTELLUNGEN
# =============================================================================
cm_to_in = 1 / 2.54
fig_width = 15.0 * cm_to_in
fig_height = 19.0 * cm_to_in  # Höhe für 3 Zeilen

USE_LATEX = False 

plt.rcParams.update({
    'text.usetex': USE_LATEX,
    'font.family': 'serif',
    # Only load the preamble if we are actually using real LaTeX
    'text.latex.preamble': r'\usepackage{lmodern} \usepackage{amsmath}' if USE_LATEX else '',
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})

# Farbpalette aus Mockup
color_frost = 'tab:olive'
color_air = 'tab:blue'
color_speed = '#8e44ad' 
color_q_cond = '#e74c3c'
color_q_evap = '#3498db'
color_pel = '#2ecc71'
color_cop = 'tab:orange'

# =============================================================================
# 3. THESIS PLOTTING LOGIC
# =============================================================================
def plot_thesis_dashboard(results, med_prop):
    # --- Echtdaten extrahieren ---
    t = np.array(results["time"])
    frost_mass = results["m_frost"]
    air_flow = results["v_dot_air"]
    comp_speed = results["speed"]
    q_cond = np.array(results["Q_cond"]) / 1000.0       # W to kW
    q_evap = np.array(results["Q_evap"]) / 1000.0       # W to kW
    p_el = np.array(results["compressor_power"]) / 1000.0 # W to kW
    cop = results["COP_wo_defrost"]

    # --- Plot aufbauen (3x2 Grid) ---
    fig, axs = plt.subplots(3, 2, figsize=(fig_width, fig_height))

    # --- ZEILE 1 ---
    # 1. Reifmasse
    axs[0, 0].plot(t, frost_mass, color=color_frost, linewidth=1.5)
    axs[0, 0].set_title('Ursache: Reifmasse [kg]')
    axs[0, 0].set_xlabel('Zeit [min]')
    # axs[0, 0].set_ylabel('Reifmasse [g]')

    # 2. Luftvolumenstrom
    axs[0, 1].plot(t, air_flow, color=color_air, linewidth=1.5)
    axs[0, 1].set_title('Luftvolumenstrom [m$^3$/h]')
    axs[0, 1].set_xlabel('Zeit [min]')
    # axs[0, 1].set_ylabel(r'Volumenstrom [m$^3$/h]')

    # --- ZEILE 2 ---
    # 3. log(p)-h Diagramm
    ax_ph = axs[1, 0]
    
    # Echter REFPROP Dome
    if med_prop is not None:
        try:
            if hasattr(med_prop, 'get_critical_point'):
                 _, p_crit, _ = med_prop.get_critical_point()
            else:
                 p_crit = 4900000.0 
            
            p_dome_range = np.logspace(np.log10(1e5), np.log10(p_crit * 0.99), 100)
            h_liq_dome, h_vap_dome, p_dome_success = [], [], []

            for p in p_dome_range:
                try:
                    hl = med_prop.calc_state("PQ", p, 0.0).h / 1000.0
                    hv = med_prop.calc_state("PQ", p, 1.0).h / 1000.0
                    h_liq_dome.append(hl)
                    h_vap_dome.append(hv)
                    p_dome_success.append(p)
                except: pass

            h_dome_full = np.concatenate([h_liq_dome, h_vap_dome[::-1]])
            p_dome_full = np.concatenate([p_dome_success, p_dome_success[::-1]])
            
            # Label entfernt, damit es nicht in der Legende auftaucht
            ax_ph.plot(h_dome_full, p_dome_full / 1e5, color='gray', linewidth=1.2)
        except Exception as e:
            print(f"Could not plot saturation dome: {e}")

    # Echte Zyklen Plotten (Start und Ende)
    cycles = results.get("cycles", [])
    cycle_start = cycles[0]
    cycle_end = cycles[-1]
    
    ax_ph.plot(np.array(cycle_start["h"])/1000.0, np.array(cycle_start["p"])/1e5, 
                color='tab:blue', marker='.', markersize=6, linewidth=1.2, label='Start')
    ax_ph.plot(np.array(cycle_end["h"])/1000.0, np.array(cycle_end["p"])/1e5, 
                color='tab:red', marker='.', markersize=6, linewidth=1.2, label='Ende')
    
    # Dynamische Limits basierend auf cycle_end
    # ax_ph.set_ylim(np.round(np.min(cycle_end["p"])/1e5 - 2), 60)
    # ax_ph.set_xlim(200, None)
        
    ax_ph.set_yscale('log')
    ax_ph.set_title('log(p)-h [bar]')
    ax_ph.set_xlabel('Enthalpie [kJ/kg]')
    # ax_ph.set_ylabel('Druck [bar]')
    
    # Legende nach oben rechts geschoben und nebeneinander (ncol=2)
    ax_ph.legend(loc='upper center', ncol=2, frameon=True, facecolor='white', framealpha=0.9, edgecolor='none', fontsize=9)

    # 4. Verdichterdrehzahl
    axs[1, 1].plot(t, comp_speed, color=color_speed, linewidth=1.5)
    axs[1, 1].set_title('Verdichterdrehzahl [Hz]')
    axs[1, 1].set_xlabel('Zeit [min]')
    # axs[1, 1].set_ylabel('Verdichterdrehzahl [Hz]')

    # --- ZEILE 3 ---
    # 5. Leistungen
    axs[2, 0].plot(t, q_cond, color=color_q_cond, linewidth=1.5, label=r'$\dot{Q}_\text{Kond}$')
    axs[2, 0].plot(t, q_evap, color=color_q_evap, linewidth=1.5, label=r'$\dot{Q}_\text{Verd}$')
    axs[2, 0].plot(t, p_el, color=color_pel, linewidth=1.5, label=r'$P_\text{el}$')
    axs[2, 0].set_title('Leistungen [kW]')
    axs[2, 0].set_xlabel('Zeit [min]')
    # axs[2, 0].set_ylabel('Leistung [kW]')
    axs[2, 0].legend(loc='best', frameon=True, facecolor='white', framealpha=0.9, edgecolor='none', fontsize=10)

    # 6. COP
    axs[2, 1].plot(t, cop, color=color_cop, linewidth=1.5)
    axs[2, 1].set_title('Resultat: COP [-]')
    axs[2, 1].set_xlabel('Zeit [min]')
    # axs[2, 1].set_ylabel('COP [-]')

    # --- Kosmetik & Layout optimieren ---
    for ax in axs.flat:
        ax.grid(True, linestyle='--', alpha=0.4, zorder=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout(pad=0.5, w_pad=1.5, h_pad=1.5)
    # plt.savefig('6_Verdampfer_Sim_Ergebnis.pdf', format='pdf', bbox_inches='tight')
    plt.show()

# =============================================================================
# 4. DATA LOADING & EXECUTION
# =============================================================================
from vclibpy.media import RefProp

REFPROP_DIR = Path(r"C:\Program Files (x86)\REFPROP")
REFPROP_DLL = REFPROP_DIR / "REFPRP64.DLL"
os.environ["RPPREFIX"] = str(REFPROP_DIR)

# med_prop = RefProp(
#     fluid_name="R32.FLD|R125.FLD",       
#     z=[0.697615, 0.302385],
#     dll_path=str(REFPROP_DLL), 
#     ref_prop_path=str(REFPROP_DIR),
#     copy_dll=False
# )

med_prop = RefProp(
    fluid_name="R134a",
    dll_path=str(REFPROP_DLL), 
    ref_prop_path=str(REFPROP_DIR),
    copy_dll=False
)


file_path = "simulation_data\grand_scop_simulation_results_abt_betta_test_10.pkl"

with open(file_path, "rb") as f:
    all_runs_data = pickle.load(f)


for i in range(7):
    target_result = all_runs_data["runs_data"][i]

    plot_thesis_dashboard(target_result, med_prop)

# Why is there A Optimum Plot

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# =============================================================================
# GLOBALE PLOT-EINSTELLUNGEN FÜR DIE MASTERARBEIT
# =============================================================================
plt.rcParams.update({
    'font.size': 11,           # Allgemeine Schriftgröße
    'axes.labelsize': 11,      # Achsenbeschriftungen
    'xtick.labelsize': 10,     # X-Achsen-Ticks
    'ytick.labelsize': 10,     # Y-Achsen-Ticks
    'legend.fontsize': 10,     # Legende
    'axes.linewidth': 1.0,     # Etwas feinere Achsenlinien
})


def main():
    # =============================================================================
    # 1. LOAD AND FILTER DATA
    # =============================================================================
    # Update this filename if you are using the _3.pkl version now
    filename = "simulation_data/grand_scop_simulation_results_abt_2.pkl"
    MANUFACTURER_PITCH_MM = 7

    try:
        with open(filename, "rb") as f:
            all_data = pickle.load(f)
            all_runs_data = all_data["runs_data"]
    except FileNotFoundError:
        print(f"Error: Could not find '{filename}'. Ensure the file is in the same directory.")
        return
    
    target_app = 'Medium'
    target_pt  = 'B'
    
    data_dict_v = {}
    data_dict_area = {}
    data_dict_cop = {}

    for run in all_runs_data:
        if "metadata" not in run or "error" in run:
            continue
            
        meta = run["metadata"]
        context = meta.get("norm_context", {})

        if context.get("application") == target_app and context.get("test_point") == target_pt:
            # UMRECHNUNG: Lamellenabstand (Pitch) von m in mm
            pitch_mm = meta["inputs"]["pitch"] * 1000.0
            
            # Extract Volume Flow Rate
            if "v_dot_air" in run:
                v_dot = np.array(run["v_dot_air"], dtype=float)
                v_dot = v_dot[~np.isnan(v_dot)]
                # UMRECHNUNG: m^3/h in m^3/s (teilen durch 3600)
                v_dot = v_dot / 3600.0
                data_dict_v[pitch_mm] = v_dot   
                
            # Extract Area
            if "total_frost_area" in run:
                area = np.array(run["total_frost_area"], dtype=float)
                area = area[~np.isnan(area)]
                data_dict_area[pitch_mm] = area

            # NEW: Extract the pre-calculated Integrated Net COP straight from the metadata
            cop = meta["inputs"]["cycle-COP"]
            data_dict_cop[pitch_mm] = cop

    if not data_dict_v or not data_dict_area or not data_dict_cop:
        print("Missing data for the target application and test point.")
        return

    # Sort data by pitch (now in mm)
    sorted_pitches = sorted(data_dict_v.keys())
    violin_data_v = [data_dict_v[p] for p in sorted_pitches]
    
    # Area is essentially constant, so we just take the mean for the line plot
    means_area = [np.mean(data_dict_area[p]) for p in sorted_pitches]
    
    # COP is a single pre-calculated value per run
    cops = [data_dict_cop[p] for p in sorted_pitches]

    # Calculate means and medians for the volume flow markers
    means_v = [np.mean(d) for d in violin_data_v]
    medians_v = [np.median(d) for d in violin_data_v]

    # =============================================================================
    # 2. PLOTTING (2-Panel Stacked Layout)
    # =============================================================================
    
    # UMRECHNUNGSFAKTOR für exakt 15.5 cm Breite
    cm_to_in = 1 / 2.54
    fig_width = 15.5 * cm_to_in
    fig_height = 12.0 * cm_to_in  # Angemessene Höhe für Thesis (anpassbar)
    
    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(fig_width, fig_height), sharex=True, gridspec_kw={'height_ratios': [1, 2]})
    ax2 = ax1.twinx()  
    
    color_orange = '#ff9933'
    color_purple = '#a682c4'
    color_cop =    'black' 

    if len(sorted_pitches) > 1:
        v_width = np.min(np.diff(sorted_pitches)) * 0.55
    else:
        v_width = 0.3 # Angepasst für mm

    # --- TOP AXIS: COP ---
    ax0.plot(sorted_pitches, cops, color=color_cop, linestyle='-', linewidth=2.0, 
             marker='o', markersize=6, markerfacecolor='white', markeredgewidth=1.5, zorder=5)
    
    # Manufacturer Vertical Line
    ax0.axvline(x=MANUFACTURER_PITCH_MM, color='dimgray', linestyle='--', linewidth=1.5, alpha=0.8, zorder=1)

    # KEIN TITEL, nur Label
    ax0.set_ylabel("Zyklus-COP", color=color_cop)
    ax0.tick_params(axis='y', colors=color_cop)
    ax0.spines['left'].set_color(color_cop)
    ax0.spines['left'].set_linewidth(1.5)
    ax0.grid(True, linestyle='--', linewidth=0.8, alpha=0.7)

    # --- BOTTOM AXIS LEFT: FULL VIOLINS (Volume Flow Rate) ---
    parts_v = ax1.violinplot(
        violin_data_v, positions=sorted_pitches, widths=v_width,
        showmeans=False, showmedians=False, showextrema=False
    )
    for pc in parts_v['bodies']:
        pc.set_facecolor(color_orange)
        pc.set_edgecolor(color_orange)
        pc.set_alpha(0.8)

    # Plot Mean and Median markers for Volume Flow
    ax1.plot(sorted_pitches, means_v, 'X', color='black', markersize=8, zorder=3)
    ax1.plot(sorted_pitches, medians_v, 'o', color='white', markeredgecolor='black', markeredgewidth=1.2, markersize=6, zorder=4)

    # Manufacturer Vertical Line
    ax1.axvline(x=MANUFACTURER_PITCH_MM, color='dimgray', linestyle='--', linewidth=1.5, alpha=0.8, zorder=1)

    # Calculate a vertical position (e.g., bottom of the data or center)
    y_min = min([np.min(d) for d in violin_data_v])

    ax1.text(MANUFACTURER_PITCH_MM - 0.02,y_min, 'Hersteller', color='dimgray', fontsize=10, fontweight='bold',
             rotation=90,ha='right', va='bottom', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1))

    # --- BOTTOM AXIS RIGHT: LINE PLOT (Total Area) ---
    ax2.plot(sorted_pitches, means_area, color=color_purple, linestyle='--', linewidth=2.0, 
             marker='D', markersize=6, markerfacecolor='white', markeredgewidth=1.5, zorder=5)

    # =============================================================================
    # 3. FORMATTING BOTTOM AXES
    # =============================================================================
    ax1.set_xlabel("Lamellenabstand in mm")
    
    # Left Y-axis (ax1 - Orange)
    ax1.set_ylabel(r"Luftvolumenstrom in m$^3$/s", color=color_orange)
    ax1.tick_params(axis='y', colors=color_orange)
    ax1.spines['left'].set_color(color_orange)
    ax1.spines['left'].set_linewidth(1.5)
    
    # Right Y-axis (ax2 - Purple)
    ax2.set_ylabel(r"Wärmeübertragende Fläche in m$^2$", color=color_purple)
    ax2.tick_params(axis='y', colors=color_purple)
    ax2.spines['right'].set_color(color_purple)
    ax2.spines['right'].set_linewidth(1.5)
    
    # Grid for bottom plot
    ax1.grid(True, linestyle='-', linewidth=0.8, alpha=0.7)
    
    # =============================================================================
    # 4. SCHÖNE LEGENDE
    # =============================================================================
    mean_marker = mlines.Line2D([], [], color='black', marker='X', linestyle='None', markersize=8, label='Mittelwert')
    median_marker = mlines.Line2D([], [], color='white', markeredgecolor='black', marker='o', linestyle='None', markersize=6, markeredgewidth=1.5, label='Median')
    
    # Legende im Plot platzieren mit leichtem Rahmen
    ax1.legend(handles=[mean_marker, median_marker], 
               loc='lower left', frameon=True, framealpha=0.9, edgecolor='#cccccc')

    # Adjust X-limits to give padding
    pad_x = v_width * 0.75
    ax1.set_xlim(min(sorted_pitches) - pad_x, max(sorted_pitches) + pad_x)

    # Ensure right axis limits show the line plot nicely without hugging the edges
    area_min, area_max = min(means_area), max(means_area)
    # Vermeide Division durch 0 bei exakt konstanter Fläche
    area_padding = (area_max - area_min) * 0.1 if area_max != area_min else area_max * 0.05
    ax2.set_ylim(area_min - area_padding, area_max + area_padding)

    # Adjust spacing between the two subplots and eliminate cutoff edges
    plt.tight_layout()
    fig.subplots_adjust(hspace=0.08) # Zieht die Graphen etwas enger zusammen
    
    plt.show()

if __name__ == "__main__":
    main()

# SCOP Calculation and Final Plot

In [2]:
# Annahmen:
# WP vollständig monovalent, kann alle benötigten Heizlasten abdecken
# Daher kein zusätzlicher Heizer benötigt
# T_biv = TOL = -10°C - Also KEIN ABSCHALTEN DER WP. DIE ARBEITET IMMER UND DECKT EIGENTLICH ALLES
# Ich berechne den SCOP_on
# Die Wärmepumpe liefert immer genau die Richtige Leistung (entweder genau die Heizlast oder die Min-/Max-Leistung)
# Ich möchte die +-10% Regel nicht beachten um es einfacher zu machen


import numpy as np
import pickle
from scipy.interpolate import interp1d

# =============================================================================
# 1. HEATING LOAD CALCULATION
# =============================================================================
def calc_heating_load(T_j: float, P_design_h: float, T_design_h: float = -10.0) -> float:
    """
    Calculates the building heating load Ph(Tj) linearly.
    The heating limit temperature is fixed at 16 °C.
    """
    T_LIMIT = 16.0
    if T_j >= T_LIMIT:
        return 0.0
    
    return P_design_h * ((T_LIMIT - T_j) / (T_LIMIT - T_design_h))

# =============================================================================
# 2. DATA EXTRACTION
# =============================================================================
def get_test_points_data(scop_data: list[dict], test_pts: dict, target_pitch: float, target_application: str) -> dict:
    """
    Extracts the 5 normative test points (A, B, C, D, E) for a specific combination.
    Returns a dictionary mapped by the outdoor temperature T_j.
    """
    points = {}
    for entry in scop_data:
        if entry["pitch"] == target_pitch and entry["application"] == target_application:
            pt_letter = entry["test_point"]
            t_air = test_pts[pt_letter]["t_air"]
            points[t_air] = {
                "COP_d": entry["cycle_COP"],
                "P_min": entry["q_min_bound"],
                "P_max": entry["q_max_bound"] # P_dh corresponds to P_max at this point
            }
    return points

# =============================================================================
# 3. NORMATIVE TEST POINTS COP CALCULATION
# =============================================================================
def calc_test_points_cop_bin(test_points_data: dict, P_design_h: float, T_design_h: float = -10.0, C_d: float = 0.9) -> tuple[list, list, list]:
    """
    Calculates the normative COP_bin for the 5 fundamental test points based on 
    whether the heat pump modulates or cycles.
    """
    t_j_points = []
    cop_bin_points = []
    p_dh_points = []
    
    for T_j, data in test_points_data.items():
        P_h_Tj = calc_heating_load(T_j, P_design_h, T_design_h)
        P_min = data["P_min"]
        COP_d = data["COP_d"]
        P_dh = data["P_max"]
        
        # Case 1: Modulation / Full Load (No cycling)
        if P_h_Tj >= P_min:
            COP_bin = COP_d
            
        # Case 2: Cycling
        else:
            CR = min(P_h_Tj / P_min, 1.0) if P_min > 0 else 1.0
            if CR == 0:
                COP_bin = 0.0
            else:
                COP_bin = COP_d * (CR / (C_d * CR + (1.0 - C_d)))
                
        t_j_points.append(T_j)
        cop_bin_points.append(COP_bin)
        p_dh_points.append(P_dh)
        
    return t_j_points, cop_bin_points, p_dh_points

# =============================================================================
# 4. INTERPOLATION FOR ALL CLIMATE BINS
# =============================================================================
def interpolate_bins(t_j_points: list, cop_bin_points: list, p_dh_points: list, climate_bins: dict) -> dict:
    """
    Interpolates and extrapolates COP_bin and P_dh for all temperatures in the heating season.
    Temperatures > 12°C are linearly extrapolated automatically.
    """
    # Sort data pairs for proper interpolation
    sorted_cop = sorted(zip(t_j_points, cop_bin_points))
    sorted_pdh = sorted(zip(t_j_points, p_dh_points))
    
    x_sorted = [p[0] for p in sorted_cop]
    y_cop = [p[1] for p in sorted_cop]
    y_pdh = [p[1] for p in sorted_pdh]
    
    cop_interpolator = interp1d(x_sorted, y_cop, kind='linear', fill_value='extrapolate')
    pdh_interpolator = interp1d(x_sorted, y_pdh, kind='linear', fill_value='extrapolate')
    
    bin_results = {}
    for T_j in sorted(climate_bins.keys()):
        if T_j >= 16.0:  # Skip bins above heating limit
            continue
            
        bin_results[T_j] = {
            "COP_bin": float(cop_interpolator(T_j)),
            "P_dh": float(pdh_interpolator(T_j))
        }
        
    return bin_results

# =============================================================================
# 5. FINAL SCOP CALCULATION
# =============================================================================
def calculate_scop_on(climate_bins: dict, bin_data: dict, P_design_h: float, T_design_h: float = -10.0, verbose: bool = True) -> float:
    """
    Sums up the thermal and electrical energy for the whole season and calculates SCOP_on.
    Strictly monovalent: Raises an error if heating load exceeds heat pump capacity.
    """
    total_eth = 0.0
    total_eel = 0.0
    
    if verbose:
        print(f"{'T_j':>4} | {'h_j':>4} | {'P_h(T_j)':>8} | {'P_dh(T_j)':>9} | {'COP_bin':>7} | {'E_th':>8} | {'E_el':>8}")
        print("-" * 75)
        
    for T_j, h_j in sorted(climate_bins.items()):
        if T_j >= 16.0:
            continue
            
        P_h_Tj = calc_heating_load(T_j, P_design_h, T_design_h)
        if P_h_Tj <= 0:
            continue
            
        COP_bin_Tj = bin_data[T_j]["COP_bin"]
        P_dh_Tj = bin_data[T_j]["P_dh"]
        
        # STRICT CHECK: Heat pump must cover the load entirely. No backup heater. - Tolerance for numerical issues: 1 W
        if P_h_Tj > (P_dh_Tj + 1.0):
            print(
                f"Capacity deficit at {T_j}°C! Heating load ({P_h_Tj:.1f} W) "
                f"exceeds heat pump max capacity ({P_dh_Tj:.1f} W). "
                f"Backup heater is disabled."
            )
            
        # Prevent division by zero or negative COPs due to extrapolation
        safe_cop = max(COP_bin_Tj, 0.1) 
        
        E_el_bin = (P_h_Tj / safe_cop) * h_j
        E_th_bin = P_h_Tj * h_j
        
        total_eth += E_th_bin
        total_eel += E_el_bin
        
        if verbose:
            print(f"{T_j:4.0f} | {h_j:4.0f} | {P_h_Tj:8.1f} | {P_dh_Tj:9.1f} | {safe_cop:7.2f} | {E_th_bin:8.1f} | {E_el_bin:8.1f}")

    SCOP_on = total_eth / total_eel if total_eel > 0 else 0.0
    
    if verbose:
        print("-" * 75)
        print(f"Total E_th: {total_eth:.1f} Wh | Total E_el: {total_eel:.1f} Wh | SCOP_on: {SCOP_on:.3f}\n")
        
    return SCOP_on


# =============================================================================
# CLIMATE DATA (DIN EN 14825)
# =============================================================================
mild_climate_bins = {
    -10: 1,   -9: 25,  -8: 23,  -7: 24,  -6: 27,  -5: 68,  -4: 91,  -3: 89,  -2: 165, -1: 173,
      0: 240,  1: 280,  2: 320,  3: 357,  4: 356,  5: 303,  6: 330,  7: 326,  8: 348,  9: 335,
     10: 315, 11: 215, 12: 169, 13: 151, 14: 105, 15: 74
}

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    DATA_FILE = "simulation_data/grand_scop_simulation_results_abt_betta_stuff.pkl"

    # 1. Load Data
    try:
        with open(DATA_FILE, "rb") as f:
            loaded_payload = pickle.load(f)
            
        all_runs_data   = loaded_payload["runs_data"]
        test_points     = loaded_payload["test_points"]
        P_DESIGNH       = loaded_payload["P_DESIGNH"]
    except FileNotFoundError:
        print(f"File {DATA_FILE} not found. Please check the path.")
        exit(1)

    # 2. Flatten Data
    scop_input_data = []
    for run in all_runs_data:
        if "error" in run:
            continue
            
        meta = run.get("metadata", {})
        inputs = meta.get("inputs", {})
        norm = meta.get("norm_context", {})
        
        scop_input_data.append({
            "pitch": inputs.get("pitch"),
            "application": norm.get("application"),
            "test_point": norm.get("test_point"),
            "q_min_bound": inputs.get("q_min_bound"),
            "q_max_bound": inputs.get("q_max_bound"),
            "cycle_COP": inputs.get("cycle-COP", inputs.get("cycle_COP"))
        })

    # 3. Identify unique combinations
    unique_combinations = set((entry["pitch"], entry["application"]) for entry in scop_input_data)
    scop_results = {}

    # 4. Iterate over combinations and execute pipeline
    for pitch, app in sorted(unique_combinations):
        print(f"=== CALCULATING SCOP FOR PITCH: {pitch}, APPLICATION: {app} ===")
        
        # Step A: Extract Data
        test_points_data = get_test_points_data(scop_input_data, test_points, pitch, app)
        
        if len(test_points_data) < 5:
            print(f"Warning: Not enough test points found for Pitch {pitch}, App {app}. Skipping...\n")
            continue
            
        # Step B: Calculate COP for the 5 normative points
        t_j_points, cop_bin_points, p_dh_points = calc_test_points_cop_bin(
            test_points_data=test_points_data, 
            P_design_h=P_DESIGNH
        )
        
        # Step C: Interpolate across all climate bins
        bin_data = interpolate_bins(
            t_j_points=t_j_points, 
            cop_bin_points=cop_bin_points, 
            p_dh_points=p_dh_points, 
            climate_bins=mild_climate_bins
        )
        
        # Step D: Final SCOP Calculation (Throws error if capacity is insufficient)
        try:
            scop_on = calculate_scop_on(
                climate_bins=mild_climate_bins,
                bin_data=bin_data,
                P_design_h=P_DESIGNH,
                verbose=True
            )
        except ValueError as e:
            print(f"FAILED: {e}\n")
            continue # Skip saving results for this invalid run
        
        # Store results
        if pitch not in scop_results:
            scop_results[pitch] = {}
        scop_results[pitch][app] = scop_on

    # 5. Save output
    final_data_structure = {
        "sim_runs": all_runs_data,
        "SCOP_on": scop_results
    }

    output_filename = DATA_FILE.replace(".pkl", "_SCOP.pkl")
    with open(output_filename, "wb") as f:
        pickle.dump(final_data_structure, f)

    print(f"Success! Calculated SCOP_on saved to: {output_filename}")

=== CALCULATING SCOP FOR PITCH: 0.001, APPLICATION: Low ===
 T_j |  h_j | P_h(T_j) | P_dh(T_j) | COP_bin |     E_th |     E_el
---------------------------------------------------------------------------
 -10 |    1 |  10400.0 |       nan |    2.70 |  10400.0 |   3857.3
  -9 |   25 |  10000.0 |       nan |    2.73 | 250000.0 |  91571.8
  -8 |   23 |   9600.0 |       nan |    2.76 | 220800.0 |  79884.1
  -7 |   24 |   9200.0 |       nan |    2.80 | 220800.0 |  78915.9
  -6 |   27 |   8800.0 |       nan |    2.83 | 237600.0 |  84013.8
  -5 |   68 |   8400.0 |       nan |    2.86 | 571200.0 | 199839.1
  -4 |   91 |   8000.0 |       nan |    2.89 | 728000.0 | 252034.7
  -3 |   89 |   7600.0 |       nan |    2.92 | 676400.0 | 231748.3
  -2 |  165 |   7200.0 |       nan |    2.95 | 1188000.0 | 402865.3
  -1 |  173 |   6800.0 |       nan |    2.98 | 1176400.0 | 394888.5
   0 |  240 |   6400.0 |       nan |    3.01 | 1536000.0 | 510424.3
   1 |  280 |   6000.0 |       nan |    3.04 | 1680000.0 